# **Cycle 4 — APIs and Reproducible Acquisition**

### *Isabella Bartoletti*

**Essential question:** How can we request live information responsibly while preserving a result that another person can reproduce?

This canonical teaching notebook is the instructional source of truth for Cycle 4.

## Teaching Notes

Use the fixture path first. Enable a live call only after students can describe the endpoint, parameters, expected schema, timeout, failure path, attribution, and usage limits. Never demonstrate a real secret in a notebook.

## Learning Objectives

Students can explain APIs and HTTP; read endpoint documentation; send a parameterized GET request; distinguish status, headers, and JSON body; recognize authentication patterns; validate a response; handle timeout/HTTP/JSON/schema failures; use a cached fallback; document provenance and respectful use; and normalize a small JSON response.

## Prior Knowledge

Cycles 1–3: functions, dictionaries/lists, exceptions, DataFrames, normalization, validation, provenance, and reproducible paths.

> **Instructor prompt:** Which parts of a live response might change tomorrow, even if our code does not?

## 1. API and HTTP Model

An API is a documented interface. A GET request combines a base endpoint, parameters, and headers. A response has a status code, headers, and body. JSON represents objects as dictionaries and arrays as lists.

In [21]:
# just get fixture and print fixture

%pip install pandas numpy

from pathlib import Path
import json
import pandas as pd
import numpy as np

#fixture = copy of API
DATA_DIR = Path("./")
FIXTURE_DIR = DATA_DIR / "api-fixtures"

assert FIXTURE_DIR.exists(), "Run from the teaching-notebook directory and keep datasets beside it."

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [22]:
fixture_path = FIXTURE_DIR / "open-meteo-nyc-forecast.json"

#file input statement
with fixture_path.open(encoding="utf-8") as handle:
    fixture = json.load(handle) # json library, imported
print(fixture.keys())
print(fixture["daily"].keys()) # data nested in json formatted data

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum'])


#### **API Intro Classwork**
 **Here, I repeat the same process of obtaining the fixture and printing the fixture for two different PokeAPI endpoints!**

 ##### 1. `/ability/battle-armor`
 - Route parameters: `ability` and `battle-armor`    
 - The data retrieved are details about the Pokemon's battle armor ability, including its name (in different languages), an explanation of the ability, the description of the ability as displayed in the game (with different versions), and more. It definitely intrigued me to see all these battle qualities compiled into a JSON file for potential data analysis.

  ##### 2. `/pokemon-species/aegislash`
 - Route parameters: `pokemon-species` and `aegislash`    
 - The data retrieved provided detailed information about Aegislash, a supposed ghost-type Pokemon I was interested in investigating! In particular, it details the Aegishlash species' characteristics, evolution, game descriptions, age status, mythicality, etc.

In [23]:
ability_path = FIXTURE_DIR / "battle-armor.json"
species_path = FIXTURE_DIR / "aegislash.json"

with ability_path.open(encoding="utf-8") as handle:
    ability = json.load(handle)

with species_path.open(encoding="utf-8") as handle:
    species = json.load(handle)

print(ability.keys())
print(ability["generation"].keys()) 

print(species.keys())
print(species["generation"].keys()) 

ENDPOINTS = [
    "https://pokeapi.co/api/v2/ability/battle-armor",
    "https://pokeapi.co/api/v2/pokemon-species/aegislash"
]

dict_keys(['id', 'name', 'is_main_series', 'generation', 'names', 'effect_entries', 'effect_changes', 'flavor_text_entries', 'pokemon'])
dict_keys(['name', 'url'])
dict_keys(['id', 'name', 'order', 'gender_rate', 'capture_rate', 'base_happiness', 'is_baby', 'is_legendary', 'is_mythical', 'hatch_counter', 'has_gender_differences', 'forms_switchable', 'growth_rate', 'pokedex_numbers', 'egg_groups', 'color', 'shape', 'evolves_from_species', 'evolution_chain', 'habitat', 'generation', 'names', 'pal_park_encounters', 'form_descriptions', 'flavor_text_entries', 'genera', 'varieties'])
dict_keys(['name', 'url'])


## 2. Read Documentation Before Requesting

For Open-Meteo `/v1/forecast`, latitude and longitude are required. We request only three daily variables, seven days, and a named timezone. Free classroom use needs no key; data attribution is required. The fixture registry records limits and retrieval details.

In [24]:
ENDPOINT = "https://api.open-meteo.com/v1/forecast" # special URLs that are doorways to accessing data
# on remote server

PARAMS = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/New_York",
    "forecast_days": 7,
}
EXPECTED_DAILY = {"time", "temperature_2m_max", "temperature_2m_min", "precipitation_sum"}

In [25]:
ENDPOINT = "https://pokeapi.co/api/v2/pokemon-species/aegislash" # special URLs that are doorways to accessing data
# on remote server



## 3. Authentication Concepts

APIs may use no credential, an API key, OAuth, or another token. Read secrets from environment variables such as `os.getenv("SERVICE_API_KEY")`; never print or commit them. A missing credential should produce a clear message and use an approved fixture when possible.

## 4. Validated Live-or-Fixture Acquisition

The default is offline. A teacher may set `use_live=True`. The function uses a timeout, checks HTTP status, parses JSON, validates schema, and falls back after any request or validation failure.

In [26]:
def validate_weather_payload(payload):
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object")
    if "daily" not in payload or not isinstance(payload["daily"], dict):
        raise ValueError("Missing daily object")
    missing = EXPECTED_DAILY - set(payload["daily"])
    if missing:
        raise ValueError(f"Missing daily fields: {sorted(missing)}")
    lengths = {key: len(payload["daily"][key]) for key in EXPECTED_DAILY}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"Daily arrays have different lengths: {lengths}")
    return payload

validate_weather_payload(fixture)

{'latitude': 40.710335,
 'longitude': -73.99308,
 'generationtime_ms': 0.07176399230957031,
 'utc_offset_seconds': -14400,
 'timezone': 'America/New_York',
 'timezone_abbreviation': 'GMT-4',
 'elevation': 32.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'precipitation_sum': 'mm'},
 'daily': {'time': ['2026-07-30',
   '2026-07-31',
   '2026-08-01',
   '2026-08-02',
   '2026-08-03',
   '2026-08-04',
   '2026-08-05'],
  'temperature_2m_max': [23.7, 30.1, 29.7, 27.4, 29.8, 25.6, 25.6],
  'temperature_2m_min': [19.5, 18.7, 20.9, 23.6, 23.2, 23.7, 23.5],
  'precipitation_sum': [0.0, 0.0, 0.0, 0.0, 17.2, 8.4, 11.4]}}

In [27]:
def acquire_weather(use_live=False):
    source = "cached fixture"
    payload = fixture
    if use_live:
        try:
            import requests
            response = requests.get(ENDPOINT, params=PARAMS, timeout=10)
            response.raise_for_status()
            payload = response.json()
            validate_weather_payload(payload)
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = fixture
    return validate_weather_payload(payload), source

weather_payload, acquisition_source = acquire_weather(use_live=False)
acquisition_source

'cached fixture'

### Intentional Bug

This example omits timeout, status checking, schema validation, and fallback. It is a pattern to critique, not copy.

In [28]:
# Fragile pattern: do not use for submitted work.
import requests
requests.get(ENDPOINT, params=PARAMS).json()

ModuleNotFoundError: No module named 'requests'

## 5. Normalize and Record Provenance

In [ ]:
daily = pd.DataFrame(weather_payload["daily"])
daily["time"] = pd.to_datetime(daily["time"])
assert len(daily) == 7
assert EXPECTED_DAILY <= set(daily.columns)
daily

,time,temperature_2m_max,temperature_2m_min,precipitation_sum
0,2026-07-30,23.7,19.5,0.0
1,2026-07-31,30.1,18.7,0.0
2,2026-08-01,29.7,20.9,0.0
3,2026-08-02,27.4,23.6,0.0
4,2026-08-03,29.8,23.2,17.2
5,2026-08-04,25.6,23.7,8.4
6,2026-08-05,25.6,23.5,11.4


In [ ]:
provenance = {
    "provider": "Open-Meteo",
    "endpoint": ENDPOINT,
    "parameters": PARAMS,
    "acquisition_source": acquisition_source,
    "fixture_retrieved": "2026-07-30",
    "license": "CC BY 4.0; attribution required",
}
provenance

{'provider': 'Open-Meteo',
 'endpoint': 'https://api.open-meteo.com/v1/forecast',
 'parameters': {'latitude': 40.7128,
  'longitude': -74.006,
  'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum',
  'timezone': 'America/New_York',
  'forecast_days': 7},
 'acquisition_source': 'cached fixture',
 'fixture_retrieved': '2026-07-30',
 'license': 'CC BY 4.0; attribution required'}

## Guided Practice

Trace endpoint → parameters → status → JSON → validation → DataFrame. Simulate a missing `daily` key and explain where the function fails. Identify the attribution and usage rule in the fixture registry.

## Independent Practice

Build an offline-first acquisition notebook containing documentation notes, parameter dictionary, timeout, failure handling, fixture fallback, schema checks, normalized DataFrame, provenance record, one exploratory observation, and reflection.

## Challenge

Add a `required_fields` parameter to the validator or support an optional environment-based API key without revealing it.

## Reflection

What changes between live and archived data? Which check prevents the most misleading failure? How does caching support respectful use and reproducibility?